[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sampriti-sr/MLP_Course/blob/kshitij-variant/docs/toolkit/sklearn-pipelines.ipynb)

# scikit-learn: Pipelines and ColumnTransformer

<div class="admonition info">
<p class="admonition-title">What you will be able to do after this page</p>
<ul>
<li>Explain and demonstrate target leakage, and prevent it by putting every learned step in a pipeline.</li>
<li>Build a ColumnTransformer with different treatments per column group.</li>
<li>Inspect a pipeline and tune it with step__parameter names.</li>
<li>Save and reload a fitted pipeline safely.</li>
</ul>
</div>

**Prerequisites:** <a href="../sklearn-preprocessing/">scikit-learn: Preprocessing</a>. **Dataset:** Credit Card Default (Taiwan, 2005). **Time:** about 45 minutes.

In [1]:
try:
    import mlp_data
except ImportError:  # e.g. a fresh Colab runtime
    %pip install -q "mlp-course @ git+https://github.com/sampriti-sr/MLP_Course@kshitij-variant"
    import mlp_data

import time, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from mlp_data import load_credit, style

style.apply()
pd.set_option("display.width", 120)
pd.set_option("display.max_columns", 14)
pd.set_option("display.float_format", "{:,.3f}".format)

## The situation

A real model is not one object, it is a **sequence**: impute the gaps, encode the categories, scale the numbers, then fit. Done by hand, each step is a chance to make one of three mistakes: **leak** information from the test data into preparation, **forget** a step when predicting on new data, or **mismatch** columns between training and prediction. A `Pipeline` fixes all three by turning the sequence into a single object that is fitted, scored and saved as one.

Same credit-default data, same split as before:

In [2]:
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.metrics import roc_auc_score

credit = load_credit()
y = credit["default payment next month"]
X = credit.drop(columns=["ID", "default payment next month"])
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=0, stratify=y)
cv = StratifiedKFold(5, shuffle=True, random_state=0)
print(X_train.shape, X_test.shape)

(22500, 23) (7500, 23)


## 1. A leak that really bites

Last page we saw that fitting a *scaler* on all the data before cross-validation made no measurable difference (a mean and standard deviation from 30,000 rows barely change). That can lull you into thinking leakage is a theoretical worry. It is not. The dangerous leaks are steps that **look at the target**, such as selecting the "best" features.

To make the effect unmistakable, we build a case where we *know* there is nothing to find: 300 customers, their real features, plus **1,000 columns of pure random noise**. Any honest model should score about 0.5 (a coin flip).

In [3]:
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

rng = np.random.default_rng(0)
small = X_train.sample(300, random_state=0)
y_small = y_train.loc[small.index]
noise = pd.DataFrame(rng.normal(size=(300, 1000)), index=small.index, columns=[f"noise_{i}" for i in range(1000)])
signal_free = noise                                                     # ONLY noise: nothing here can predict the target

# WRONG: choose the 20 'best' columns using ALL the labels, then cross-validate on those 20
chosen = SelectKBest(f_classif, k=20).fit(signal_free, y_small).transform(signal_free)
wrong = cross_val_score(LogisticRegression(max_iter=2000), chosen, y_small, cv=cv, scoring="roc_auc").mean()

# RIGHT: selection is a pipeline step, so it is refitted inside each fold using only that fold's training rows
right = cross_val_score(make_pipeline(SelectKBest(f_classif, k=20), LogisticRegression(max_iter=2000)), signal_free, y_small, cv=cv, scoring="roc_auc").mean()

print(f"selection OUTSIDE cross-validation: AUC {wrong:.3f}   (an impossibly good score for random noise)")
print(f"selection INSIDE a pipeline:        AUC {right:.3f}   (about a coin flip, which is the truth)")
assert wrong > right + 0.1

selection OUTSIDE cross-validation: AUC 0.854   (an impossibly good score for random noise)
selection INSIDE a pipeline:        AUC 0.504   (about a coin flip, which is the truth)


The wrong way reports a respectable AUC *from random noise*. The selection step peeked at all the labels, picked the 20 noise columns that happened to correlate with them by chance, and the cross-validation then "confirmed" those columns on the very rows that chose them. Inside a pipeline the selection is redone from scratch in every fold on the training rows only, and the score collapses to the truth. **The rule: anything that learns from the data, and especially from the target, belongs inside the pipeline.**

## 2. Different columns, different treatment: `ColumnTransformer`

Our columns need different preparation: amounts need scaling, repayment status should be one-hot encoded, and `SEX`, `EDUCATION`, `MARRIAGE` are codes for categories. `ColumnTransformer` applies the right transformer to each group of columns and glues the results together.

In [4]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder

categorical = ["SEX", "EDUCATION", "MARRIAGE"] + [c for c in X.columns if c.startswith("PAY_") and not c.startswith("PAY_AMT")]
numeric = [c for c in X.columns if c not in categorical]
print(f"{len(categorical)} categorical columns, {len(numeric)} numeric columns")

preprocess = ColumnTransformer([
    ("categories", OneHotEncoder(handle_unknown="ignore"), categorical),
    ("amounts", StandardScaler(), numeric),
])
pipe = make_pipeline(preprocess, LogisticRegression(max_iter=3000))
print(pipe)

9 categorical columns, 14 numeric columns
Pipeline(steps=[('columntransformer',
                 ColumnTransformer(transformers=[('categories',
                                                  OneHotEncoder(handle_unknown='ignore'),
                                                  ['SEX', 'EDUCATION',
                                                   'MARRIAGE', 'PAY_0', 'PAY_2',
                                                   'PAY_3', 'PAY_4', 'PAY_5',
                                                   'PAY_6']),
                                                 ('amounts', StandardScaler(),
                                                  ['LIMIT_BAL', 'AGE',
                                                   'BILL_AMT1', 'BILL_AMT2',
                                                   'BILL_AMT3', 'BILL_AMT4',
                                                   'BILL_AMT5', 'BILL_AMT6',
                                                   'PAY_AMT1', 'PAY_AMT2',
                  

Read it from the outside in: a pipeline of two steps (the column transformer, then the classifier), where the column transformer has two branches. The steps get automatic names (`columntransformer`, `logisticregression`) which we will need for tuning. Now it behaves like any single model:

In [5]:
scores = cross_val_score(pipe, X_train, y_train, cv=cv, scoring="roc_auc")
pipe.fit(X_train, y_train)
test_auc = roc_auc_score(y_test, pipe.predict_proba(X_test)[:, 1])
print(f"cross-validated AUC {scores.mean():.3f} (+/- {scores.std():.3f});  AUC on the untouched test set {test_auc:.3f}")

cross-validated AUC 0.772 (+/- 0.008);  AUC on the untouched test set 0.759


One call to `cross_val_score` re-fits the encoder, the scaler *and* the classifier inside each fold, so there is no leak by construction. And `pipe.predict(new_rows)` automatically applies the same encoding and scaling to new data: nothing to forget.

## 3. Looking inside

Pipelines are not black boxes. Index into them by position or name, and ask the preprocessing step what columns it produced:

In [6]:
names = pipe[:-1].get_feature_names_out()                       # everything except the final model
print(f"{X_train.shape[1]} input columns became {len(names)} model inputs after encoding")

coef = pd.Series(pipe[-1].coef_[0], index=names)
strongest = coef.reindex(coef.abs().sort_values(ascending=False).index).head(8)
strongest.rename(lambda n: n.replace("categories__", "").replace("amounts__", "")).to_frame("coefficient")

23 input columns became 90 model inputs after encoding


,coefficient
PAY_0_0,-1.321
PAY_0_-2,-1.145
PAY_0_3,0.957
PAY_0_2,0.920
EDUCATION_5,-0.781
PAY_2_4,-0.658
MARRIAGE_0,-0.653
PAY_5_4,-0.636


The feature names let us read the model in terms of the original columns instead of anonymous column numbers. The top entries are mostly repayment-status levels: on-time statuses push the predicted risk down, two or three months late pushes it up. But look at the other entries: **rare levels appear among the largest weights, and one has the wrong sign.** `PAY_2_4` (four months late last-but-one month) is held by only 99 customers, about half of whom default, yet its coefficient is *negative*; `EDUCATION_5` is an undocumented code held by 280 customers; `MARRIAGE_0` by 54. Rare categories get large, unstable coefficients (and with correlated repayment columns the model can trade weight between them), which is exactly why the preprocessing page offered `min_frequency`. Never interpret a single coefficient on a rare level.

## 4. Tuning a pipeline: the `step__parameter` rule

To tune anything inside a pipeline, name it `stepname__parametername` (double underscore). Steps nested inside a step chain further: `columntransformer__amounts__with_mean`. Name your steps explicitly to keep these readable:

In [7]:
from sklearn.model_selection import GridSearchCV

from sklearn.pipeline import Pipeline

named = Pipeline([("prep", ColumnTransformer([("cats", OneHotEncoder(handle_unknown="ignore"), categorical), ("nums", StandardScaler(), numeric)])),
                  ("clf", LogisticRegression(max_iter=3000))])

grid = GridSearchCV(named, param_grid={"clf__C": [0.001, 0.01, 0.1, 1, 10]}, cv=cv, scoring="roc_auc", n_jobs=-1)
grid.fit(X_train, y_train)

results = pd.DataFrame(grid.cv_results_)[["param_clf__C", "mean_test_score", "std_test_score"]].rename(columns={"param_clf__C": "C", "mean_test_score": "mean CV AUC", "std_test_score": "sd"})
print("best:", grid.best_params_, f"| best CV AUC {grid.best_score_:.4f} | test AUC {roc_auc_score(y_test, grid.predict_proba(X_test)[:, 1]):.4f}")
results

best: {'clf__C': 0.1} | best CV AUC 0.7725 | test AUC 0.7589


,C,mean CV AUC,sd
0,0.001,0.766,0.005
1,0.010,0.772,0.006
2,0.100,0.772,0.008
3,1.000,0.772,0.008
4,10.000,0.772,0.008


`GridSearchCV` tries every combination, scoring each by cross-validation *on the training data*, then **refits the best on all training data** and exposes it as `grid.predict` etc. The test set is touched once, at the end. Notice how flat the table is: from `C = 0.01` upwards the AUC is the same to three decimals, and only the strongest penalty (`0.001`) is a little worse, by about 0.006, roughly one fold-to-fold standard deviation (`sd`). When a grid is this flat the model is insensitive to the setting; do not read the "best" row as meaningful.

## 5. Saving and loading

A fitted pipeline is one object, so persisting it persists *everything*: the encoder, scaler and classifier together.

In [8]:
import joblib, tempfile, os
path = os.path.join(tempfile.mkdtemp(), "credit_pipeline.joblib")
joblib.dump(grid.best_estimator_, path)
reloaded = joblib.load(path)
assert np.allclose(reloaded.predict_proba(X_test)[:, 1], grid.predict_proba(X_test)[:, 1])
print(f"saved and reloaded ({os.path.getsize(path) / 1024:.0f} KB); predictions identical")

saved and reloaded (6 KB); predictions identical


<div class="admonition danger">
<p class="admonition-title">Only load model files you trust</p>
<p>A joblib file is a pickle, and loading a pickle can execute arbitrary code. The lifecycle chapter's packaging page shows how to store a model with a hash and verify it before loading.</p>
</div>

## Pitfalls at a glance

| Surprise | Cause | Fix |
|---|---|---|
| CV score looks too good to be true | a step that sees the labels ran outside the CV | put **every** learned step in the pipeline |
| `ValueError: Invalid parameter` in a grid | wrong name | `pipe.get_params().keys()`; use `step__param` |
| New data fails at predict time | columns differ from training | use a `ColumnTransformer` with named columns |
| Columns silently dropped | `ColumnTransformer` drops unlisted columns by default | `remainder="passthrough"` to keep them |
| Grid search is slow | many combinations x many folds | fewer values, `n_jobs=-1`, or `RandomizedSearchCV` |

## Check yourself

<details class="question">
<summary>1. Why did selecting features before cross-validation produce an impressive AUC from pure noise?</summary>

The selection step used *all* the labels to pick the columns that happened to correlate with them. The CV then evaluated those columns on the same rows that had chosen them, so the "test" was not independent. Inside a pipeline, selection is redone in each fold using only that fold's training rows, so the held-out fold cannot influence it.

</details>

<details class="question">
<summary>2. What does <code>ColumnTransformer</code> do with columns you do not mention?</summary>

By default it **drops them**. Set `remainder="passthrough"` to keep them unchanged, or give them their own transformer. Forgetting this is a common reason a feature seems to have no effect: it never reached the model.

</details>

<details class="question">
<summary>3. In a grid search, how do you refer to the <code>C</code> of a step named <code>clf</code>?</summary>

`clf__C`: the step name, two underscores, then the parameter. For nested steps, chain them, e.g. `prep__nums__with_mean`. `pipe.get_params().keys()` lists every valid name.

</details>

## Exercises

<details class="question">
<summary>Insert a <code>SimpleImputer(strategy="median")</code> for the numeric columns and check the pipeline still runs when 10% of the amounts are missing.</summary>

```python
from sklearn.impute import SimpleImputer
robust = Pipeline([("prep", ColumnTransformer([
    ("cats", OneHotEncoder(handle_unknown="ignore"), categorical),
    ("nums", make_pipeline(SimpleImputer(strategy="median"), StandardScaler()), numeric)])),
    ("clf", LogisticRegression(max_iter=3000))])

holes = X_train.copy()
holes.loc[holes.sample(frac=0.1, random_state=0).index, "BILL_AMT1"] = np.nan
print(cross_val_score(robust, holes, y_train, cv=cv, scoring="roc_auc").mean().round(3))   # about 0.772, the same as with no gaps
```

A pipeline nests: the numeric branch is itself a small pipeline (impute, then scale). Without the imputer, the same call would fail with a `NaN` error from the scaler or classifier.

</details>

<details class="question">
<summary>Use <code>RandomizedSearchCV</code> to tune both <code>clf__C</code> (on a log scale) and whether the encoder uses <code>handle_unknown="ignore"</code> or <code>"infrequent_if_exist"</code>. Why is random search a good fit for wide grids?</summary>

```python
from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import loguniform
search = RandomizedSearchCV(named, {"clf__C": loguniform(1e-3, 1e1), "prep__cats__handle_unknown": ["ignore", "infrequent_if_exist"]},
                            n_iter=8, cv=cv, scoring="roc_auc", random_state=0, n_jobs=-1)
```

Random search samples a fixed number of settings from distributions, so its cost does not explode as you add parameters, and it covers each parameter's range more evenly than a coarse grid. Given how flat the AUC was across `C`, expect little to separate the candidates: across eight random settings the best and worst cross-validated AUC differ by under 0.005.

</details>

## Recap

- A **Pipeline** chains steps into one object, so cross-validation refits everything per fold, `predict` repeats every step, and saving keeps it all together.
- **Leakage from steps that see the target** (feature selection, target encoding) is large and real; leakage from a plain scaler is often tiny. Put *all* learned steps in the pipeline regardless.
- `ColumnTransformer` gives each column group its own treatment; unlisted columns are **dropped** unless you pass them through.
- Tune with `step__parameter` names via `GridSearchCV` or `RandomizedSearchCV`; the test set is used once, at the end.
- A flat grid means the setting barely matters.

**Next:** *Models*, the main families and when each fits.